In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/datdigitaladdict/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

print("Working dir:", os.getcwd())

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded:", "yes" if hf_token else "no")

Working dir: /content/flyrank-ml-internship
Token loaded: yes


In [2]:
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=hf_token)
for f in sorted(files)[:40]:
    print(f)
print(f"\n...{len(files)} files total")

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [3]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

hf_base = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{hf_base}/fact_content_daily_performance/month=2026-03/data_0.parquet"

test = con.execute(f"SELECT COUNT(*) AS row_count FROM read_parquet('{month_path}')").df()
print(test)

   row_count
0    9841378


In [4]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{month_path}') LIMIT 1").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datdigitaladdict/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My data contract for Lane 2 (Refresh / Content Opportunity Scoring):

1. Unit of analysis + time window: One row is one content item, for one client, on one day. I am developing on the mid-panel month month=2026-03 from fact_content_daily_performance, treating the final month (fact_content_daily_performance_sample, June 2026) as a sealed test month I will not touch during development.
2. Table(s): fact_content_daily_performance (the daily fact table) joined to dim_content on content_hash_id, and dim_clients on client_hash_id when client-level context is needed.
3. Time window: March 2026 (month=2026-03) for feature and label development, avoiding the final month to prevent peeking at a future outcome window.
4. Target or proxy: A declining-visibility proxy, built from a drop in gsc_impressions or gsc_clicks compared to an earlier window. This still needs a precise threshold before use as a real label.
5. One thing I deliberately exclude: any rebuilt product decision flag such as health_score or priority_score. These are not in this data by design, and I will not reconstruct them, since doing so would let a model copy an existing decision instead of learning from evidence.

In [5]:
row_count = con.execute(f"SELECT COUNT(*) AS row_count FROM read_parquet('{month_path}')").df()
print(row_count)


   row_count
0    9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Sorting fields into four buckets:

Feature candidates: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, scroll_events.

Label/proxy: a declining-visibility flag I will build from gsc_impressions or gsc_clicks trending downward across report dates.

Context (used for joins/grouping, not as model inputs): report_date, client_hash_id, content_hash_id, gsc_data_available, ga4_data_available.

Excluded: any rebuilt health_score, priority_score, or action_type, because these are the product's own decisions and would let a model copy an existing answer rather than learn from evidence. Also excluded: any raw query, URL, or title text, since none is present in this pseudonymized release.



In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as n
    FROM read_parquet('{month_path}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain rows found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,n


In [8]:
span = con.execute(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{month_path}')
""").df()
print(span)

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


In [9]:
availability = con.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as gsc_available_rows
    FROM read_parquet('{month_path}')
""").df()
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  gsc_available_rows
0     9841378              413966             3611061


My five features for this month, each with why it is knowable at the decision moment:

1. gsc_impressions - knowable because it is a completed record of past search visibility, measured before any reviewer looks at the page.
2. gsc_clicks - knowable for the same reason, a completed past measurement.
3. gsc_avg_position - knowable because it reflects where the page already ranked in the past, not a future outcome.
4. ga4_sessions - knowable because it is a completed count of past visits.
5. scroll_events - knowable because it is a completed count of past on-page behavior.

In [10]:
features = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) as gsc_impressions,
        SUM(gsc_clicks) as gsc_clicks,
        AVG(gsc_avg_position) as gsc_avg_position,
        SUM(ga4_sessions) as ga4_sessions,
        SUM(scroll_events) as scroll_events
    FROM read_parquet('{month_path}')
    GROUP BY content_hash_id, client_hash_id
""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 7)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,2.714744,0.0,0.0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,1.0,0.0
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,6.481453,4.0,0.0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,2.987198,0.0,0.0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,3.0,0.0


Now I will deliberately add a label-derived column to show the leakage trap. I will build a simple "declining" label from clicks, then add a column that is literally derived from that same label, and watch a quick score jump toward perfect then remove it.

In [11]:
import numpy as np

# Build a simple declining label: bottom 50% of clicks this month
median_clicks = features["gsc_clicks"].median()
features["is_declining"] = (features["gsc_clicks"] <= median_clicks).astype(int)

print("Label distribution:")
print(features["is_declining"].value_counts())

Label distribution:
is_declining
1    262600
0     68837
Name: count, dtype: int64


In [12]:
# THE DELIBERATE LEAK: a column derived directly from the label itself
features["leaky_column"] = features["is_declining"] * 100

from sklearn.metrics import roc_auc_score

honest_score = roc_auc_score(features["is_declining"], features["gsc_clicks"])
leaky_score = roc_auc_score(features["is_declining"], features["leaky_column"])

print(f"Honest score using gsc_clicks alone (AUC): {honest_score:.3f}")
print(f"Leaky score using leaky_column (AUC): {leaky_score:.3f}   <- suspiciously perfect")

Honest score using gsc_clicks alone (AUC): 0.000
Leaky score using leaky_column (AUC): 1.000   <- suspiciously perfect


In [13]:
features = features.drop(columns=["leaky_column"])
print("Leaky column removed. Columns remaining:", list(features.columns))
print(f"Honest score stands at: {honest_score:.3f}")

Leaky column removed. Columns remaining: ['content_hash_id', 'client_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events', 'is_declining']
Honest score stands at: 0.000


Note: the "honest" score of 0.000 is not a bug, it reflects that I defined is_declining directly from a threshold on gsc_clicks itself, so of course they are perfectly (inversely) related. This is itself a smaller version of the same lesson: a label built directly from one of your candidate features is not independent evidence, it is circular by construction. A more honest label for future work would come from a genuinely separate signal or a future time window, not a threshold on the same column being used as a feature.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One real limitation of this data: client tracking history is unbalanced. Not every client's gsc_data_available and ga4_data_available start on the same date, so early rows for some clients may only have search data and no analytics data yet. A single month's snapshot also cannot show seasonality, since it has no comparison to the same period in a prior year.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.